<a href="https://colab.research.google.com/github/intanelaqsha/Python-Learn/blob/main/event_aqsha.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## GEOM

In [24]:
"""
Grievance MHGID Processing Pipeline
Pure Python/Pandas translation of the original DuckDB SQL script.
Requires: pandas, geopandas, shapely, pyarrow, pyproj
"""

import re
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from pyproj import Transformer
import networkx as nx

# ============================================================
# HELPERS
# ============================================================

SOC_ISSUES = [
    'Corruption', 'Forced Labor', 'Gender and Ethnic Disparities',
    'Human Rights Violation', 'Land Dispute', 'Indigenous Peoples Conflict',
    'Labor Disputes', 'Labor Rights Violations', 'Land Grabbing',
    'Limited Access to Services', 'Violence and/or Coercion', 'Wage Dispute',
]

ENV_ISSUES = [
    'Deforestation', 'Fires', 'Peatland Loss', 'Biodiversity loss',
    'Illegal Infrastructure', 'Infrastructure Damage', 'Riparian Issues',
    'Environmental Pollution',
]

def has_any(text, keywords):
    """Return True if text contains any of the keywords."""
    if not isinstance(text, str):
        return False
    t = text.strip()
    return any(kw in t for kw in keywords)

def classify_issue(issues):
    """Replicate the CASE WHEN issue_category logic."""
    is_soc = has_any(issues, SOC_ISSUES)
    is_env = has_any(issues, ENV_ISSUES)
    if is_soc and is_env:
        return 'SOC_ENV'
    elif is_soc:
        return 'SOC_ONLY'
    elif is_env:
        return 'ENV_ONLY'
    else:
        return 'SOC_ENV'  # default

def clean_classify(val):
    """Strip whitespace; return None if only whitespace/non-printable chars."""
    if pd.isna(val):
        return None
    # Remove all whitespace/non-printable
    stripped_all = re.sub(r'\s', '', str(val))
    if stripped_all == '':
        return None
    return str(val).strip()

def split_ids(series):
    """
    Expand a comma-separated ID column into individual rows.
    Returns a Series aligned to original index with one ID per entry.
    """
    return series.str.split(',')

def to_int_safe(val):
    try:
        return int(str(val).strip())
    except (ValueError, TypeError):
        return None


# ============================================================
# STEP 1: Load raw tables
# ============================================================

print("Step 1: Loading raw tables...")

grievances = pd.read_csv('Grievances-Grid view.csv', dtype=str, on_bad_lines='skip')
grievances = grievances.rename(columns={
    'ID': 'grievance_id',
    'Source': 'source',
    'Suppliers': 'suppliers',
    'Responsible Company': 'responsible_company',
    'Issues': 'issues',
    'PIOConcessions-v2': 'pioid',
    'Mills': 'umlid',
    'Classify': 'classify_raw',
})
grievances['classify'] = grievances['classify_raw'].apply(clean_classify)
grievances = grievances[['grievance_id', 'source', 'suppliers', 'responsible_company',
                          'issues', 'pioid', 'umlid', 'classify']]

plots = pd.read_csv('Concessions-v2-Grid view.csv', dtype=str, on_bad_lines='skip')
plots = plots.rename(columns={
    'ID': 'pioid',
    'Name': 'plot_name',
    'Group': 'plot_group',
})
plots['pioid'] = pd.to_numeric(plots['pioid'], errors='coerce')
plots = plots[['pioid', 'plot_name', 'plot_group']]

mills_csv = pd.read_csv('Mills.csv', dtype=str, on_bad_lines='skip')
mills = mills_csv.rename(columns={
    'UML_ID': 'umlid',
    'Mill_Name': 'mill_name',
    'Group': 'mill_group',
})
mills = mills[['umlid', 'mill_name', 'mill_group']]


# ============================================================
# STEP 1B: Load geometry
# ============================================================

print("Step 1B: Loading geometries...")

transformer_4326_to_3857 = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)

# plots_geom: polygon centroids in EPSG:3857
plots_gdf = gpd.read_parquet('plots_v2_20260417.parquet')
print("Parquet columns:", plots_gdf.columns.tolist())
print("Parquet CRS:", plots_gdf.crs)

# Auto-detect geometry column (geopandas sets .geometry automatically,
# but fall back to finding any geometry-typed column if needed)
geom_col = plots_gdf.geometry.name  # geopandas active geometry column
id_col_candidates = [c for c in plots_gdf.columns if c.upper() == 'PIOID']
if not id_col_candidates:
    raise KeyError(f"No PIOID column found. Available columns: {plots_gdf.columns.tolist()}")
id_col = id_col_candidates[0]

plots_gdf = plots_gdf[[id_col, geom_col]].copy()
plots_gdf = plots_gdf.rename(columns={id_col: 'pioid'})
plots_gdf = plots_gdf.set_geometry(geom_col)
plots_gdf['pioid'] = pd.to_numeric(plots_gdf['pioid'], errors='coerce')

# Reproject to EPSG:3857
if plots_gdf.crs is None or plots_gdf.crs.to_epsg() != 4326:
    plots_gdf = plots_gdf.set_crs("EPSG:4326", allow_override=True)
plots_gdf_3857 = plots_gdf.to_crs("EPSG:3857")

# Use ORIGINAL polygon geometry
plots_geom = plots_gdf_3857[['pioid', geom_col]].copy()
plots_geom = plots_geom.rename(columns={geom_col: 'geom'})
plots_geom = plots_geom.reset_index(drop=True)

# mills_geom: point geometry in EPSG:3857
mills_with_coords = mills_csv[mills_csv['Latitude'].notna() & mills_csv['Longitude'].notna()].copy()
mills_with_coords['Latitude'] = pd.to_numeric(mills_with_coords['Latitude'], errors='coerce')
mills_with_coords['Longitude'] = pd.to_numeric(mills_with_coords['Longitude'], errors='coerce')
mills_with_coords = mills_with_coords.dropna(subset=['Latitude', 'Longitude'])

def make_point_3857(row):
    x, y = transformer_4326_to_3857.transform(row['Longitude'], row['Latitude'])
    return Point(x, y)

mills_with_coords['geom'] = mills_with_coords.apply(make_point_3857, axis=1)
mills_geom = mills_with_coords[['UML_ID', 'geom']].rename(columns={'UML_ID': 'umlid'})


# ============================================================
# STEP 2: Separate grievances WITH and WITHOUT classify
# ============================================================

print("Step 2: Splitting grievances by classify...")

grievances_with_classify = grievances[grievances['classify'].notna() &
                                       (grievances['classify'].str.strip() != '')].copy()
grievances_without_classify = grievances[grievances['classify'].isna() |
                                          (grievances['classify'].str.strip() == '')].copy()


# ============================================================
# HELPER: Expand a grievance table by splitting pioid / umlid
# ============================================================

def expand_pioid(df):
    """Expand comma-separated pioid into one row per pioid."""
    rows = []
    for _, row in df.iterrows():
        pios = [p.strip() for p in str(row['pioid']).split(',') if p.strip()] if pd.notna(row['pioid']) else ['']
        for pio in pios:
            r = row.to_dict()
            r['_pioid_val'] = pio
            rows.append(r)
    return pd.DataFrame(rows)

def expand_umlid(df):
    """Expand comma-separated umlid into one row per umlid."""
    rows = []
    for _, row in df.iterrows():
        umls = [u.strip() for u in str(row['umlid']).split(',') if u.strip()] if pd.notna(row['umlid']) else ['']
        for uml in umls:
            r = row.to_dict()
            r['_umlid_val'] = uml
            rows.append(r)
    return pd.DataFrame(rows)


# ============================================================
# STEP 3A: Expand plot & mill for grievances WITH classify
# ============================================================

print("Step 3A: Expanding classify rows...")

# -- pioid branch --
gc_pio = expand_pioid(grievances_with_classify)
gc_pio['_pioid_int'] = gc_pio['_pioid_val'].apply(to_int_safe)
gc_pio = gc_pio.merge(plots, left_on='_pioid_int', right_on='pioid', how='left', suffixes=('', '_p'))

classify_pio = pd.DataFrame({
    'grievance_id': gc_pio['grievance_id'],
    'source': gc_pio['source'],
    'suppliers': gc_pio['suppliers'],
    'responsible_company': gc_pio['responsible_company'],
    'issues': gc_pio['issues'].str.strip(),
    'classify': gc_pio['classify'].str.strip(),
    'pioid': gc_pio['_pioid_int'],
    'plot_name': gc_pio['plot_name'],
    'plot_group': gc_pio['plot_group'],
    'umlid': None,
    'mill_name': None,
    'mill_group': None,
})

# -- umlid branch --
gc_uml = expand_umlid(grievances_with_classify)
gc_uml['_umlid_val'] = gc_uml['_umlid_val'].str.strip()
gc_uml = gc_uml.merge(mills, left_on='_umlid_val', right_on='umlid', how='left', suffixes=('', '_m'))

classify_uml = pd.DataFrame({
    'grievance_id': gc_uml['grievance_id'],
    'source': gc_uml['source'],
    'suppliers': gc_uml['suppliers'],
    'responsible_company': gc_uml['responsible_company'],
    'issues': gc_uml['issues'].str.strip(),
    'classify': gc_uml['classify'].str.strip(),
    'pioid': None,
    'plot_name': None,
    'plot_group': None,
    'umlid': gc_uml['_umlid_val'],
    'mill_name': gc_uml['mill_name'],
    'mill_group': gc_uml['mill_group'],
})

classify_full = pd.concat([classify_pio, classify_uml], ignore_index=True)

# mhgid_classify: CLASSIFY_ALL
mhgid_classify = (
    classify_full[classify_full['classify'].notna() & (classify_full['classify'].str.strip() != '')]
    [['grievance_id', 'classify']]
    .drop_duplicates()
    .assign(mhgid=lambda d: d['classify'].str.strip() + '_ALL')
    [['grievance_id', 'mhgid']]
    .drop_duplicates()
)


# ============================================================
# STEP 3B: Expand plot & mill for grievances WITHOUT classify
# ============================================================

print("Step 3B: Expanding no-classify rows...")

def build_grievance_full_branch_pio(df, plots, plots_geom):
    exp = expand_pioid(df)
    exp['_pioid_int'] = exp['_pioid_val'].apply(to_int_safe)
    exp = exp.merge(plots, left_on='_pioid_int', right_on='pioid', how='left', suffixes=('', '_p'))
    exp = exp.merge(plots_geom, left_on='_pioid_int', right_on='pioid', how='left', suffixes=('', '_gc'))
    exp['issue_category'] = exp['issues'].apply(classify_issue)
    exp['group_key'] = exp['responsible_company'].fillna('UNKNOWN')
    return pd.DataFrame({
        'grievance_id': exp['grievance_id'],
        'source': exp['source'],
        'suppliers': exp['suppliers'],
        'responsible_company': exp['responsible_company'],
        'issues': exp['issues'].str.strip(),
        'classify': exp['classify'],
        'pioid': exp['_pioid_int'],
        'plot_name': exp['plot_name'],
        'plot_group': exp['plot_group'],
        'umlid': None,
        'mill_name': None,
        'mill_group': None,
        'geom': exp['geom'],
        'issue_category': exp['issue_category'],
        'group_key': exp['group_key'],
    })

def build_grievance_full_branch_uml(df, mills, mills_geom):
    exp = expand_umlid(df)
    exp['_umlid_val'] = exp['_umlid_val'].str.strip()
    exp = exp.merge(mills, left_on='_umlid_val', right_on='umlid', how='left', suffixes=('', '_m'))
    exp = exp.merge(mills_geom, left_on='_umlid_val', right_on='umlid', how='left', suffixes=('', '_mg'))
    exp['issue_category'] = exp['issues'].apply(classify_issue)
    exp['group_key'] = exp['responsible_company'].fillna('UNKNOWN')
    return pd.DataFrame({
        'grievance_id': exp['grievance_id'],
        'source': exp['source'],
        'suppliers': exp['suppliers'],
        'responsible_company': exp['responsible_company'],
        'issues': exp['issues'].str.strip(),
        'classify': exp['classify'],
        'pioid': None,
        'plot_name': None,
        'plot_group': None,
        'umlid': exp['_umlid_val'],
        'mill_name': exp['mill_name'],
        'mill_group': exp['mill_group'],
        'geom': exp['geom'],
        'issue_category': exp['issue_category'],
        'group_key': exp['group_key'],
    })

gf_pio = build_grievance_full_branch_pio(
    grievances_without_classify,
    plots,
    plots_geom
)
gf_uml = build_grievance_full_branch_uml(grievances_without_classify, mills, mills_geom)
grievance_full = pd.concat([gf_pio, gf_uml], ignore_index=True)


# ============================================================
# STEP 4: SOC grouping (SOC_ONLY → per responsible_company)
# ============================================================

print("Step 4: SOC grouping...")

mhgid_soc = (
    grievance_full[grievance_full['issue_category'] == 'SOC_ONLY']
    [['grievance_id', 'responsible_company']]
    .drop_duplicates()
    .assign(mhgid=lambda d: d['responsible_company'].fillna('UNKNOWN') + '_SOC')
    [['grievance_id', 'mhgid']]
    .drop_duplicates()
)

soc_grievance_ids = set(mhgid_soc['grievance_id'])


# ============================================================
# STEP 5: ENV proximity — 100km cluster
# ============================================================

# ============================================================
# STEP 5: ENV proximity — TRUE 100km polygon clustering
# ============================================================

print("Step 5: ENV proximity clustering (100km polygon-edge)...")

env_rows = grievance_full[
    grievance_full['geom'].notna() &
    ~grievance_full['grievance_id'].isin(soc_grievance_ids)
].copy()

env_rows['node_id'] = (
    env_rows['grievance_id'].astype(str) + '_' +
    env_rows.apply(
        lambda r: str(int(r['pioid'])) if pd.notna(r['pioid']) else str(r['umlid']),
        axis=1
    )
)

env_node_ids = set(env_rows['grievance_id'])

RADIUS_M = 100_000

clusters = []

for company in env_rows['responsible_company'].dropna().unique():

    company_rows = env_rows[
        env_rows['responsible_company'] == company
    ].reset_index(drop=True)

    if len(company_rows) == 0:
        continue

    geoms = list(company_rows['geom'])

    # spatial index
    tree = STRtree(geoms)

    # graph
    G = nx.Graph()

    for idx, row in company_rows.iterrows():

        node_id = row['node_id']
        geom = row['geom']

        G.add_node(node_id)

        # candidate polygons within buffer
        candidate_idxs = tree.query(geom.buffer(RADIUS_M))

        for cand_idx in candidate_idxs:

            if idx == cand_idx:
                continue

            cand_row = company_rows.iloc[cand_idx]

            cand_node = cand_row['node_id']
            cand_geom = cand_row['geom']

            # POLYGON EDGE DISTANCE
            if geom.distance(cand_geom) <= RADIUS_M:

                G.add_edge(node_id, cand_node)

    # connected components = TRUE cluster
    connected = list(nx.connected_components(G))

    for rank, component in enumerate(connected, start=1):

        mhgid = f"{company}_100km_{rank}"

        for node in component:

            grievance_id = node.split('_')[0]

            clusters.append({
                'node_id': node,
                'grievance_id': grievance_id,
                'responsible_company': company,
                'mhgid': mhgid
            })

mhgid_env = pd.DataFrame(clusters)

# ============================================================
# STEP 6: Grievances without any plot/mill at all (no geom)
# ============================================================

print("Step 6: No plot/mill fallback...")

grievance_no_plotmill = grievance_full[
    ~grievance_full['grievance_id'].isin(soc_grievance_ids) &
    ~grievance_full['grievance_id'].isin(env_node_ids)
].copy()

def no_plotmill_mhgid(row):
    rc = row['responsible_company'] if pd.notna(row['responsible_company']) else 'UNKNOWN'
    if row['issue_category'] == 'SOC_ONLY':
        return rc + '_SOC'
    else:
        return rc + '_no_plotmill'

mhgid_no_plotmill = (
    grievance_no_plotmill[['grievance_id', 'issue_category', 'responsible_company']]
    .drop_duplicates()
    .copy()
)
mhgid_no_plotmill['mhgid'] = mhgid_no_plotmill.apply(no_plotmill_mhgid, axis=1)
mhgid_no_plotmill = mhgid_no_plotmill[['grievance_id', 'mhgid']].drop_duplicates()


# ============================================================
# STEP 7: Final Union
# Priority: 1.CLASSIFY → 2.SOC → 3.ENV → 4.NO_PLOTMILL
# ============================================================

print("Step 7: Final union...")

OUTPUT_COLS = [
    'mhgid', 'grievance_id', 'source', 'suppliers', 'responsible_company',
    'pioid', 'plot_name', 'plot_group', 'umlid', 'mill_name', 'mill_group', 'issues'
]

# 1. CLASSIFY
part1 = classify_full.merge(mhgid_classify, on='grievance_id', how='inner')
part1 = part1[OUTPUT_COLS]

# 2. SOC
part2 = grievance_full.merge(mhgid_soc, on='grievance_id', how='inner')
part2 = part2[OUTPUT_COLS]

# 3. ENV
env_node_map = mhgid_env[['node_id', 'mhgid']].drop_duplicates()
gf_with_node = grievance_full[~grievance_full['grievance_id'].isin(soc_grievance_ids)].copy()
gf_with_node['node_id'] = (
    gf_with_node['grievance_id'].astype(str) + '_' +
    gf_with_node.apply(
        lambda r: str(int(r['pioid'])) if pd.notna(r['pioid']) else str(r['umlid']),
        axis=1
    )
)
part3 = gf_with_node.merge(env_node_map, on='node_id', how='inner')
part3 = part3[OUTPUT_COLS]

# 4. NO_PLOTMILL
no_plotmill_gids = set(grievance_no_plotmill['grievance_id'])
part4 = grievance_full[
    ~grievance_full['grievance_id'].isin(soc_grievance_ids) &
    ~grievance_full['grievance_id'].isin(env_node_ids)
].merge(mhgid_no_plotmill, on='grievance_id', how='inner')
part4 = part4[OUTPUT_COLS]

grievance_mhgid_all = pd.concat([part1, part2, part3, part4], ignore_index=True)
grievance_mhgid_all = grievance_mhgid_all[grievance_mhgid_all['mhgid'].notna()]


# ============================================================
# STEP 8: Grouped output
# ============================================================

print("Step 8: Grouping output...")

def agg_distinct(series):
    vals = pd.unique(series.dropna())

    cleaned = []
    for v in vals:
        # convert float-like IDs → int string
        if isinstance(v, (float, int)):
            if pd.notna(v):
                if float(v).is_integer():
                    cleaned.append(str(int(v)))
                else:
                    cleaned.append(str(v))
        else:
            s = str(v).strip()

            # remove np.float64(...)
            s = re.sub(r'np\.float64\((.*?)\)', r'\1', s)

            cleaned.append(s)

    return ', '.join(cleaned)

grievance_mhgid_grouped = (
    grievance_mhgid_all
    .groupby('mhgid', as_index=False)
    .agg(
        grievance_count=('grievance_id', 'nunique'),
        grievance_ids=('grievance_id', agg_distinct),
        sources=('source', agg_distinct),
        suppliers=('suppliers', agg_distinct),
        responsible_companies=('responsible_company', agg_distinct),
        pioids=('pioid', agg_distinct),
        plot_names=('plot_name', agg_distinct),
        plot_groups=('plot_group', agg_distinct),
        umlids=('umlid', agg_distinct),
        mill_names=('mill_name', agg_distinct),
        mill_groups=('mill_group', agg_distinct),
        issues=('issues', agg_distinct),
    )
)

# EXPORT
grievance_mhgid_grouped.to_csv('grievance_responsible_geom.csv', index=False)
print("Done! Output written to grievance_responsible_geom.csv")
print(f"Total mhgid groups: {len(grievance_mhgid_grouped)}")

Step 1: Loading raw tables...
Step 1B: Loading geometries...
Parquet columns: ['fid', 'PIOID', 'Name', 'Group', 'Country', 'geom']
Parquet CRS: OGC:CRS84
Step 2: Splitting grievances by classify...
Step 3A: Expanding classify rows...
Step 3B: Expanding no-classify rows...


/tmp/ipykernel_323/1213763589.py:249: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  classify_full = pd.concat([classify_pio, classify_uml], ignore_index=True)
/tmp/ipykernel_323/1213763589.py:324: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  grievance_full = pd.concat([gf_pio, gf_uml], ignore_index=True)


Step 4: SOC grouping...
Step 5: ENV proximity clustering (100km polygon-edge)...
Step 6: No plot/mill fallback...
Step 7: Final union...
Step 8: Grouping output...
Done! Output written to grievance_responsible_centroid.csv
Total mhgid groups: 567


what is next


*   ubah logic chain. jika A ke B, B ke C, C ke D. D within max 150km ke A or B so they can connect.C also should be within 150km to A.
*  kenapa cuma ngubah jadi geom, clusternya langsung berantakan?


In [4]:
"""
Grievance MHGID Processing Pipeline
Pure Python/Pandas translation of the original DuckDB SQL script.
Requires: pandas, geopandas, shapely, pyarrow, pyproj
"""

import re
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from pyproj import Transformer
import networkx as nx
from itertools import combinations
# ============================================================
# HELPERS
# ============================================================

SOC_ISSUES = [
    'Corruption', 'Forced Labor', 'Gender and Ethnic Disparities',
    'Human Rights Violation', 'Land Dispute', 'Indigenous Peoples Conflict',
    'Labor Disputes', 'Labor Rights Violations', 'Land Grabbing',
    'Limited Access to Services', 'Violence and/or Coercion', 'Wage Dispute',
]

ENV_ISSUES = [
    'Deforestation', 'Fires', 'Peatland Loss', 'Biodiversity loss',
    'Illegal Infrastructure', 'Infrastructure Damage', 'Riparian Issues',
    'Environmental Pollution',
]

def has_any(text, keywords):
    """Return True if text contains any of the keywords."""
    if not isinstance(text, str):
        return False
    t = text.strip()
    return any(kw in t for kw in keywords)

def classify_issue(issues):
    """Replicate the CASE WHEN issue_category logic."""
    is_soc = has_any(issues, SOC_ISSUES)
    is_env = has_any(issues, ENV_ISSUES)
    if is_soc and is_env:
        return 'SOC_ENV'
    elif is_soc:
        return 'SOC_ONLY'
    elif is_env:
        return 'ENV_ONLY'
    else:
        return 'SOC_ENV'  # default

def clean_classify(val):
    """Strip whitespace; return None if only whitespace/non-printable chars."""
    if pd.isna(val):
        return None
    # Remove all whitespace/non-printable
    stripped_all = re.sub(r'\s', '', str(val))
    if stripped_all == '':
        return None
    return str(val).strip()

def split_ids(series):
    """
    Expand a comma-separated ID column into individual rows.
    Returns a Series aligned to original index with one ID per entry.
    """
    return series.str.split(',')

def to_int_safe(val):
    try:
        return int(str(val).strip())
    except (ValueError, TypeError):
        return None


# ============================================================
# STEP 1: Load raw tables
# ============================================================

print("Step 1: Loading raw tables...")

grievances = pd.read_csv('Grievances-Grid view.csv', dtype=str, on_bad_lines='skip')
grievances = grievances.rename(columns={
    'ID': 'grievance_id',
    'Source': 'source',
    'Suppliers': 'suppliers',
    'Responsible Company': 'responsible_company',
    'Issues': 'issues',
    'PIOConcessions-v2': 'pioid',
    'Mills': 'umlid',
    'Classify': 'classify_raw',
})
grievances['classify'] = grievances['classify_raw'].apply(clean_classify)
grievances = grievances[['grievance_id', 'source', 'suppliers', 'responsible_company',
                          'issues', 'pioid', 'umlid', 'classify']]

plots = pd.read_csv('Concessions-v2-Grid view.csv', dtype=str, on_bad_lines='skip')
plots = plots.rename(columns={
    'ID': 'pioid',
    'Name': 'plot_name',
    'Group': 'plot_group',
})
plots['pioid'] = pd.to_numeric(plots['pioid'], errors='coerce')
plots = plots[['pioid', 'plot_name', 'plot_group']]

mills_csv = pd.read_csv('Mills.csv', dtype=str, on_bad_lines='skip')
mills = mills_csv.rename(columns={
    'UML_ID': 'umlid',
    'Mill_Name': 'mill_name',
    'Group': 'mill_group',
})
mills = mills[['umlid', 'mill_name', 'mill_group']]


# ============================================================
# STEP 1B: Load geometry
# ============================================================

print("Step 1B: Loading geometries...")

transformer_4326_to_3857 = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)

# plots_geom: polygon centroids in EPSG:3857
plots_gdf = gpd.read_parquet('plots_v2_20260417.parquet')
print("Parquet columns:", plots_gdf.columns.tolist())
print("Parquet CRS:", plots_gdf.crs)

# Auto-detect geometry column (geopandas sets .geometry automatically,
# but fall back to finding any geometry-typed column if needed)
geom_col = plots_gdf.geometry.name  # geopandas active geometry column
id_col_candidates = [c for c in plots_gdf.columns if c.upper() == 'PIOID']
if not id_col_candidates:
    raise KeyError(f"No PIOID column found. Available columns: {plots_gdf.columns.tolist()}")
id_col = id_col_candidates[0]

plots_gdf = plots_gdf[[id_col, geom_col]].copy()
plots_gdf = plots_gdf.rename(columns={id_col: 'pioid'})
plots_gdf = plots_gdf.set_geometry(geom_col)
plots_gdf['pioid'] = pd.to_numeric(plots_gdf['pioid'], errors='coerce')

# Reproject to EPSG:3857
if plots_gdf.crs is None or plots_gdf.crs.to_epsg() != 4326:
    plots_gdf = plots_gdf.set_crs("EPSG:4326", allow_override=True)
plots_gdf_3857 = plots_gdf.to_crs("EPSG:3857")

# Use ORIGINAL polygon geometry
plots_geom = plots_gdf_3857[['pioid', geom_col]].copy()
plots_geom = plots_geom.rename(columns={geom_col: 'geom'})
plots_geom = plots_geom.reset_index(drop=True)

# mills_geom: point geometry in EPSG:3857
mills_with_coords = mills_csv[mills_csv['Latitude'].notna() & mills_csv['Longitude'].notna()].copy()
mills_with_coords['Latitude'] = pd.to_numeric(mills_with_coords['Latitude'], errors='coerce')
mills_with_coords['Longitude'] = pd.to_numeric(mills_with_coords['Longitude'], errors='coerce')
mills_with_coords = mills_with_coords.dropna(subset=['Latitude', 'Longitude'])

def make_point_3857(row):
    x, y = transformer_4326_to_3857.transform(row['Longitude'], row['Latitude'])
    return Point(x, y)

mills_with_coords['geom'] = mills_with_coords.apply(make_point_3857, axis=1)
mills_geom = mills_with_coords[['UML_ID', 'geom']].rename(columns={'UML_ID': 'umlid'})


# ============================================================
# STEP 2: Separate grievances WITH and WITHOUT classify
# ============================================================

print("Step 2: Splitting grievances by classify...")

grievances_with_classify = grievances[grievances['classify'].notna() &
                                       (grievances['classify'].str.strip() != '')].copy()
grievances_without_classify = grievances[grievances['classify'].isna() |
                                          (grievances['classify'].str.strip() == '')].copy()


# ============================================================
# HELPER: Expand a grievance table by splitting pioid / umlid
# ============================================================

def expand_pioid(df):
    """Expand comma-separated pioid into one row per pioid."""
    rows = []
    for _, row in df.iterrows():
        pios = [p.strip() for p in str(row['pioid']).split(',') if p.strip()] if pd.notna(row['pioid']) else ['']
        for pio in pios:
            r = row.to_dict()
            r['_pioid_val'] = pio
            rows.append(r)
    return pd.DataFrame(rows)

def expand_umlid(df):
    """Expand comma-separated umlid into one row per umlid."""
    rows = []
    for _, row in df.iterrows():
        umls = [u.strip() for u in str(row['umlid']).split(',') if u.strip()] if pd.notna(row['umlid']) else ['']
        for uml in umls:
            r = row.to_dict()
            r['_umlid_val'] = uml
            rows.append(r)
    return pd.DataFrame(rows)


# ============================================================
# STEP 3A: Expand plot & mill for grievances WITH classify
# ============================================================

print("Step 3A: Expanding classify rows...")

# -- pioid branch --
gc_pio = expand_pioid(grievances_with_classify)
gc_pio['_pioid_int'] = gc_pio['_pioid_val'].apply(to_int_safe)
gc_pio = gc_pio.merge(plots, left_on='_pioid_int', right_on='pioid', how='left', suffixes=('', '_p'))

classify_pio = pd.DataFrame({
    'grievance_id': gc_pio['grievance_id'],
    'source': gc_pio['source'],
    'suppliers': gc_pio['suppliers'],
    'responsible_company': gc_pio['responsible_company'],
    'issues': gc_pio['issues'].str.strip(),
    'classify': gc_pio['classify'].str.strip(),
    'pioid': gc_pio['_pioid_int'],
    'plot_name': gc_pio['plot_name'],
    'plot_group': gc_pio['plot_group'],
    'umlid': None,
    'mill_name': None,
    'mill_group': None,
})

# -- umlid branch --
gc_uml = expand_umlid(grievances_with_classify)
gc_uml['_umlid_val'] = gc_uml['_umlid_val'].str.strip()
gc_uml = gc_uml.merge(mills, left_on='_umlid_val', right_on='umlid', how='left', suffixes=('', '_m'))

classify_uml = pd.DataFrame({
    'grievance_id': gc_uml['grievance_id'],
    'source': gc_uml['source'],
    'suppliers': gc_uml['suppliers'],
    'responsible_company': gc_uml['responsible_company'],
    'issues': gc_uml['issues'].str.strip(),
    'classify': gc_uml['classify'].str.strip(),
    'pioid': None,
    'plot_name': None,
    'plot_group': None,
    'umlid': gc_uml['_umlid_val'],
    'mill_name': gc_uml['mill_name'],
    'mill_group': gc_uml['mill_group'],
})

classify_full = pd.concat([classify_pio, classify_uml], ignore_index=True)

# mhgid_classify: CLASSIFY_ALL
mhgid_classify = (
    classify_full[classify_full['classify'].notna() & (classify_full['classify'].str.strip() != '')]
    [['grievance_id', 'classify']]
    .drop_duplicates()
    .assign(mhgid=lambda d: d['classify'].str.strip() + '_ALL')
    [['grievance_id', 'mhgid']]
    .drop_duplicates()
)


# ============================================================
# STEP 3B: Expand plot & mill for grievances WITHOUT classify
# ============================================================

print("Step 3B: Expanding no-classify rows...")

def build_grievance_full_branch_pio(df, plots, plots_geom):
    exp = expand_pioid(df)
    exp['_pioid_int'] = exp['_pioid_val'].apply(to_int_safe)
    exp = exp.merge(plots, left_on='_pioid_int', right_on='pioid', how='left', suffixes=('', '_p'))
    exp = exp.merge(plots_geom, left_on='_pioid_int', right_on='pioid', how='left', suffixes=('', '_gc'))
    exp['issue_category'] = exp['issues'].apply(classify_issue)
    exp['group_key'] = exp['responsible_company'].fillna('UNKNOWN')
    return pd.DataFrame({
        'grievance_id': exp['grievance_id'],
        'source': exp['source'],
        'suppliers': exp['suppliers'],
        'responsible_company': exp['responsible_company'],
        'issues': exp['issues'].str.strip(),
        'classify': exp['classify'],
        'pioid': exp['_pioid_int'],
        'plot_name': exp['plot_name'],
        'plot_group': exp['plot_group'],
        'umlid': None,
        'mill_name': None,
        'mill_group': None,
        'geom': exp['geom'],
        'issue_category': exp['issue_category'],
        'group_key': exp['group_key'],
    })

def build_grievance_full_branch_uml(df, mills, mills_geom):
    exp = expand_umlid(df)
    exp['_umlid_val'] = exp['_umlid_val'].str.strip()
    exp = exp.merge(mills, left_on='_umlid_val', right_on='umlid', how='left', suffixes=('', '_m'))
    exp = exp.merge(mills_geom, left_on='_umlid_val', right_on='umlid', how='left', suffixes=('', '_mg'))
    exp['issue_category'] = exp['issues'].apply(classify_issue)
    exp['group_key'] = exp['responsible_company'].fillna('UNKNOWN')
    return pd.DataFrame({
        'grievance_id': exp['grievance_id'],
        'source': exp['source'],
        'suppliers': exp['suppliers'],
        'responsible_company': exp['responsible_company'],
        'issues': exp['issues'].str.strip(),
        'classify': exp['classify'],
        'pioid': None,
        'plot_name': None,
        'plot_group': None,
        'umlid': exp['_umlid_val'],
        'mill_name': exp['mill_name'],
        'mill_group': exp['mill_group'],
        'geom': exp['geom'],
        'issue_category': exp['issue_category'],
        'group_key': exp['group_key'],
    })

gf_pio = build_grievance_full_branch_pio(
    grievances_without_classify,
    plots,
    plots_geom
)
gf_uml = build_grievance_full_branch_uml(grievances_without_classify, mills, mills_geom)
grievance_full = pd.concat([gf_pio, gf_uml], ignore_index=True)


# ============================================================
# STEP 4: SOC grouping (SOC_ONLY → per responsible_company)
# ============================================================

print("Step 4: SOC grouping...")

mhgid_soc = (
    grievance_full[grievance_full['issue_category'] == 'SOC_ONLY']
    [['grievance_id', 'responsible_company']]
    .drop_duplicates()
    .assign(mhgid=lambda d: d['responsible_company'].fillna('UNKNOWN') + '_SOC')
    [['grievance_id', 'mhgid']]
    .drop_duplicates()
)

soc_grievance_ids = set(mhgid_soc['grievance_id'])


# ============================================================
# STEP 5: ENV compact clustering (max 100km within cluster)
# ============================================================

print("Step 5: ENV compact polygon clustering...")

env_rows = grievance_full[
    grievance_full['geom'].notna() &
    ~grievance_full['grievance_id'].isin(soc_grievance_ids)
].copy()

env_rows['node_id'] = (
    env_rows['grievance_id'].astype(str) + '_' +
    env_rows.apply(
        lambda r: str(int(r['pioid'])) if pd.notna(r['pioid']) else str(r['umlid']),
        axis=1
    )
)

env_node_ids = set(env_rows['grievance_id'])

RADIUS_M = 120_000

clusters = []

for company in env_rows['responsible_company'].dropna().unique():

    company_rows = env_rows[
        env_rows['responsible_company'] == company
    ].reset_index(drop=True)

    if len(company_rows) == 0:
        continue

    # unassigned polygons
    remaining = set(company_rows.index)

    company_cluster_id = 1

    while remaining:

        # seed cluster
        seed_idx = remaining.pop()

        cluster_members = [seed_idx]

        added = True

        while added:

            added = False

            candidates = list(remaining)

            for cand_idx in candidates:

                cand_geom = company_rows.loc[cand_idx, 'geom']

                # candidate must be within 100km
                # to ALL existing cluster members
                valid = True

                for member_idx in cluster_members:

                    member_geom = company_rows.loc[member_idx, 'geom']

                    dist = cand_geom.distance(member_geom)

                    if dist > RADIUS_M:
                        valid = False
                        break

                if valid:
                    cluster_members.append(cand_idx)
                    remaining.remove(cand_idx)
                    added = True

        # save cluster
        mhgid = f"{company}_100km_{company_cluster_id}"

        for idx in cluster_members:

            row = company_rows.loc[idx]

            clusters.append({
                'node_id': row['node_id'],
                'grievance_id': row['grievance_id'],
                'responsible_company': company,
                'mhgid': mhgid
            })

        company_cluster_id += 1

mhgid_env = pd.DataFrame(clusters)

# ============================================================
# STEP 6: Grievances without any plot/mill at all (no geom)
# ============================================================

print("Step 6: No plot/mill fallback...")

grievance_no_plotmill = grievance_full[
    ~grievance_full['grievance_id'].isin(soc_grievance_ids) &
    ~grievance_full['grievance_id'].isin(env_node_ids)
].copy()

def no_plotmill_mhgid(row):
    rc = row['responsible_company'] if pd.notna(row['responsible_company']) else 'UNKNOWN'
    if row['issue_category'] == 'SOC_ONLY':
        return rc + '_SOC'
    else:
        return rc + '_no_plotmill'

mhgid_no_plotmill = (
    grievance_no_plotmill[['grievance_id', 'issue_category', 'responsible_company']]
    .drop_duplicates()
    .copy()
)
mhgid_no_plotmill['mhgid'] = mhgid_no_plotmill.apply(no_plotmill_mhgid, axis=1)
mhgid_no_plotmill = mhgid_no_plotmill[['grievance_id', 'mhgid']].drop_duplicates()


# ============================================================
# STEP 7: Final Union
# Priority: 1.CLASSIFY → 2.SOC → 3.ENV → 4.NO_PLOTMILL
# ============================================================

print("Step 7: Final union...")

OUTPUT_COLS = [
    'mhgid', 'grievance_id', 'source', 'suppliers', 'responsible_company',
    'pioid', 'plot_name', 'plot_group', 'umlid', 'mill_name', 'mill_group', 'issues'
]

# 1. CLASSIFY
part1 = classify_full.merge(mhgid_classify, on='grievance_id', how='inner')
part1 = part1[OUTPUT_COLS]

# 2. SOC
part2 = grievance_full.merge(mhgid_soc, on='grievance_id', how='inner')
part2 = part2[OUTPUT_COLS]

# 3. ENV
env_node_map = mhgid_env[['node_id', 'mhgid']].drop_duplicates()
gf_with_node = grievance_full[~grievance_full['grievance_id'].isin(soc_grievance_ids)].copy()
gf_with_node['node_id'] = (
    gf_with_node['grievance_id'].astype(str) + '_' +
    gf_with_node.apply(
        lambda r: str(int(r['pioid'])) if pd.notna(r['pioid']) else str(r['umlid']),
        axis=1
    )
)
part3 = gf_with_node.merge(env_node_map, on='node_id', how='inner')
part3 = part3[OUTPUT_COLS]

# 4. NO_PLOTMILL
no_plotmill_gids = set(grievance_no_plotmill['grievance_id'])
part4 = grievance_full[
    ~grievance_full['grievance_id'].isin(soc_grievance_ids) &
    ~grievance_full['grievance_id'].isin(env_node_ids)
].merge(mhgid_no_plotmill, on='grievance_id', how='inner')
part4 = part4[OUTPUT_COLS]

grievance_mhgid_all = pd.concat([part1, part2, part3, part4], ignore_index=True)
grievance_mhgid_all = grievance_mhgid_all[grievance_mhgid_all['mhgid'].notna()]


# ============================================================
# STEP 8: Grouped output
# ============================================================

print("Step 8: Grouping output...")

def agg_distinct(series):
    vals = pd.unique(series.dropna())

    cleaned = []
    for v in vals:
        # convert float-like IDs → int string
        if isinstance(v, (float, int)):
            if pd.notna(v):
                if float(v).is_integer():
                    cleaned.append(str(int(v)))
                else:
                    cleaned.append(str(v))
        else:
            s = str(v).strip()

            # remove np.float64(...)
            s = re.sub(r'np\.float64\((.*?)\)', r'\1', s)

            cleaned.append(s)

    return ', '.join(cleaned)

grievance_mhgid_grouped = (
    grievance_mhgid_all
    .groupby('mhgid', as_index=False)
    .agg(
        grievance_count=('grievance_id', 'nunique'),
        grievance_ids=('grievance_id', agg_distinct),
        sources=('source', agg_distinct),
        suppliers=('suppliers', agg_distinct),
        responsible_companies=('responsible_company', agg_distinct),
        pioids=('pioid', agg_distinct),
        plot_names=('plot_name', agg_distinct),
        plot_groups=('plot_group', agg_distinct),
        umlids=('umlid', agg_distinct),
        mill_names=('mill_name', agg_distinct),
        mill_groups=('mill_group', agg_distinct),
        issues=('issues', agg_distinct),
    )
)

# EXPORT
grievance_mhgid_grouped.to_csv('grievance_responsible_geom.csv', index=False)
print("Done! Output written to grievance_responsible_geom.csv")
print(f"Total mhgid groups: {len(grievance_mhgid_grouped)}")

Step 1: Loading raw tables...
Step 1B: Loading geometries...
Parquet columns: ['fid', 'PIOID', 'Name', 'Group', 'Country', 'geom']
Parquet CRS: OGC:CRS84
Step 2: Splitting grievances by classify...
Step 3A: Expanding classify rows...
Step 3B: Expanding no-classify rows...


/tmp/ipykernel_1568/699822147.py:249: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  classify_full = pd.concat([classify_pio, classify_uml], ignore_index=True)
/tmp/ipykernel_1568/699822147.py:324: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  grievance_full = pd.concat([gf_pio, gf_uml], ignore_index=True)


Step 4: SOC grouping...
Step 5: ENV compact polygon clustering...
Step 6: No plot/mill fallback...
Step 7: Final union...
Step 8: Grouping output...
Done! Output written to grievance_responsible_geom.csv
Total mhgid groups: 594


What have been changed


*   replace centorid into geom (polygon edges)
*   fixed spatial grouping chain
*   no eliminated grievance



